In [1]:
# CELL 1 — Setup. Always safe to run. Run this first every session.
import pandas as pd
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY")
print("Setup done. Key loaded:", api_key is not None)

Setup done. Key loaded: True


In [2]:
# CELL 2 — Load the prompts from disk (no regeneration, no API call)
prompts_df = pd.read_json("nab_prompts.json", orient="records")
print("Prompts loaded:", prompts_df.shape[0], "across", prompts_df["product_family"].nunique(), "families")

Prompts loaded: 32 across 8 families


In [3]:
# CELL 3 — Load the gather results from disk (no re-gathering, no API call)
results_df = pd.read_json("nab_credit_cards_results.json", orient="records")
print("Results loaded:", results_df.shape[0], "rows")

Results loaded: 20 rows


In [4]:
# Inspect actual answers — trust the data before building analysis
import textwrap

for _, row in results_df.sort_values("model_requested").iterrows():
    print("=" * 80)
    print(f"{row['model_requested']}   (actual: {row['model_actual']})")
    print(f"Q: {row['question']}")
    print(f"mentioned={row['brand_mentioned']}   citations={row['num_citations']}")
    print("-" * 80)
    print(textwrap.fill(str(row['answer'])[:600], width=80))
    cites = row['citations'] if isinstance(row['citations'], list) else []
    if cites:
        print(f"\n  cites: {cites[:3]}")
    print()

anthropic/claude-opus-5:online   (actual: anthropic/claude-opus-5)
Q: I keep paying high interest on my credit card in Australia and want to find a better deal.
mentioned=True   citations=21
--------------------------------------------------------------------------------
I'll look up what's currently available in the Australian market.Good news:
there's a fair bit of competition right now, and one timing detail worth knowing
about.  ## First, the context  Australians owe around $21.5 billion on credit
cards accruing interest at an average standard rate of 20.99% p.a. With the RBA
holding the cash rate at 4.35% in August and cuts not expected until mid-2027,
waiting for rates to fall isn't a strategy. So switching is the lever you
control.  ## Two different fixes (pick based on your situation)  **1. A 0%
balance transfer — if you have a chunk of debt you can re

  cites: ['https://www.mozo.com.au/credit-cards/balance-transfer', 'https://www.mozo.com.au/credit-cards/balance-transfer', 'h

In [5]:
for model in results_df["model_requested"].unique():
    sub = results_df[results_df["model_requested"] == model]
    all_cites = [c for lst in sub["citations"] if isinstance(lst, list) for c in lst]
    sample = all_cites[0] if all_cites else "(none)"
    print(f"{model:35s} {len(all_cites):>3} cites | {sample[:70]}")

openai/gpt-5.6-luna:online           16 cites | https://www.canstar.com.au/credit-cards/compare/balance-transfers/?utm
anthropic/claude-sonnet-5:online     58 cites | https://www.mozo.com.au/best/best-credit-cards
anthropic/claude-opus-5:online       87 cites | https://www.mozo.com.au/credit-cards/balance-transfer
google/gemini-3.7-flash:online        5 cites | https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQF
perplexity/sonar                     40 cites | https://www.canstar.com.au/credit-cards/compare/low-rate-credit-cards/


In [6]:
import re

BRAND = re.compile(r'\bnab\b|national australia bank', re.IGNORECASE)

def mentions(text):
    text = str(text)
    return [text[max(0, m.start()-45):m.end()+45].replace("\n", " ")
            for m in BRAND.finditer(text)]

results_df["mentioned_v2"] = results_df["answer"].apply(lambda a: len(mentions(a)) > 0)

disagree = results_df[results_df["brand_mentioned"] != results_df["mentioned_v2"]]
print(f"Disagreements old vs new: {len(disagree)}")
print(disagree[["model_requested", "question", "brand_mentioned", "mentioned_v2"]].to_string())

print("\n--- every match in context ---")
for _, r in results_df.iterrows():
    hits = mentions(r["answer"])
    if hits:
        print(f"\n{r['model_requested']} | {r['question'][:50]}")
        for h in hits:
            print(f"   …{h}…")

Disagreements old vs new: 0
Empty DataFrame
Columns: [model_requested, question, brand_mentioned, mentioned_v2]
Index: []

--- every match in context ---

anthropic/claude-opus-5:online | I keep paying high interest on my credit card in A
   … Latitude 28° Global Platinum Mastercard and NAB Low Rate, both at 0% for 12 months with a 3%…

google/gemini-3.7-flash:online | What should I look for when comparing credit cards
   …rams:** (e.g., CommBank Awards, ANZ Rewards, NAB Rewards, Westpac Altitude) allowing flexibil…

anthropic/claude-opus-5:online | Are rewards credit cards worth it compared to low-
   …rds program value. - It's already happening: NAB, ANZ and Westpac have announced changes to t…

anthropic/claude-sonnet-5:online | Can I apply for a low-rate credit card online in A
   …e whole process digitally. For example, with NAB you simply check that you meet the eligibili…

google/gemini-3.7-flash:online | Can I apply for a low-rate credit card online in A
   …  Many major banks 

In [7]:
# one Gemini row's full raw payload — did search fire but annotations drop?
g = results_df[(results_df["model_requested"]=="google/gemini-3.7-flash:online") &
               (results_df["num_citations"]==0)].iloc[0]
print(g["question"])
print("keys:", list(g.index))
print("error:", g.get("error"))
# and check what you actually stored — do you keep the raw API response anywhere,
# or only the extracted citations list?

I keep paying high interest on my credit card in Australia and want to find a better deal.
keys: ['model_requested', 'model_actual', 'brand_mentioned', 'answer', 'citations', 'num_citations', 'error', 'funnel_stage', 'question', 'product_family', 'mentioned_v2']
error: nan


In [8]:
import os, json, requests
from dotenv import load_dotenv
load_dotenv()

q = "I keep paying high interest on my credit card in Australia and want to find a better deal."

r = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
    json={"model": "google/gemini-3.7-flash:online",
          "messages": [{"role": "user", "content": q}]},
)
data = r.json()
print("status:", r.status_code)

# dump the full structure so nothing hides
print(json.dumps(data, indent=2)[:4000])

status: 200
{
  "id": "gen-1788928264-6Saw7I103pWKps1CinVz",
  "object": "chat.completion",
  "created": 1788928264,
  "model": "google/gemini-3.7-flash",
  "provider": "Google",
  "system_fingerprint": null,
  "service_tier": "default",
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "STOP",
      "message": {
        "role": "assistant",
        "content": "If you are carrying a balance on an Australian credit card, standard interest rates are often between **19% and 25% p.a.**, which means a large portion of your monthly repayment is just paying interest rather than reducing the principal.\n\nHere are the best strategies and options available in Australia to get a better deal and stop losing money to interest:\n\n---\n\n### 1. The Fastest Fix: 0% Balance Transfer Card\nA **Balance Transfer (BT)** card allows you to move your existing debt to a new credit card with a promotional **0% p.a. interest rate** for 

In [9]:
results_df.shape

(20, 11)

In [10]:
results_df[results_df['num_citations'] > 0].shape[0]

17

In [11]:
searched = results_df[results_df['num_citations'] > 0]
memory   = results_df[results_df['num_citations'] == 0]

In [12]:
memory[['model_requested', 'brand_mentioned', 'question']]

,model_requested,brand_mentioned,question
3,google/gemini-3.7-flash:online,False,I keep paying high interest on my credit card ...
8,google/gemini-3.7-flash:online,True,What should I look for when comparing credit c...
13,google/gemini-3.7-flash:online,False,Are rewards credit cards worth it compared to ...


In [13]:
results_df[(results_df['brand_mentioned']) & (results_df['num_citations'] > 0)]

,model_requested,model_actual,brand_mentioned,answer,citations,num_citations,error,funnel_stage,question,product_family,mentioned_v2
2,anthropic/claude-opus-5:online,anthropic/claude-opus-5,True,I'll look up what's currently available in the...,[https://www.mozo.com.au/credit-cards/balance-...,21,NaN,problem_aware,I keep paying high interest on my credit card ...,Credit cards,True
12,anthropic/claude-opus-5:online,anthropic/claude-opus-5,True,I'll look up current Australian credit card da...,[https://www.mozo.com.au/best/best-credit-card...,13,NaN,comparison,Are rewards credit cards worth it compared to ...,Credit cards,True
16,anthropic/claude-sonnet-5:online,anthropic/claude-sonnet-5,True,"Yes, absolutely — you can apply for a low-rate...",[https://www.americanexpress.com/en-au/credit-...,12,NaN,bottom_funnel,Can I apply for a low-rate credit card online ...,Credit cards,True
18,google/gemini-3.7-flash:online,google/gemini-3.7-flash,True,"**Yes, you can apply online in Australia for a...",[https://vertexaisearch.cloud.google.com/groun...,5,NaN,bottom_funnel,Can I apply for a low-rate credit card online ...,Credit cards,True


In [14]:
from urllib.parse import urlparse

all_urls = results_df['citations'].explode().dropna()
print(f"{len(all_urls)} citation URLs across all 20 calls\n")
print(all_urls.value_counts().head(10))

206 citation URLs across all 20 calls

citations
https://www.money.com.au/credit-cards                                    26
https://www.mozo.com.au/credit-cards                                     15
https://www.mozo.com.au/best/best-credit-cards                           12
https://worthitcards.com.au/balance-transfer-credit-cards-australia/      9
https://www.money.com.au/credit-cards/low-interest-credit-cards           9
https://www.finder.com.au/credit-cards                                    8
https://www.canstar.com.au/credit-cards/                                  8
https://www.mozo.com.au/credit-cards/balance-transfer                     7
https://www.anz.com.au/personal/credit-cards/compare-cards/               7
https://www.finder.com.au/credit-cards/low-interest-rate-credit-cards     6
Name: count, dtype: int64


In [15]:
def get_domain(url):
    return urlparse(url).netloc.replace('www.', '')

domains = all_urls.apply(get_domain)
domains.value_counts().head(15)

citations
money.com.au                       41
mozo.com.au                        38
canstar.com.au                     24
finder.com.au                      24
moneysmart.gov.au                  10
anz.com.au                         10
worthitcards.com.au                 9
creditcard.com.au                   9
vertexaisearch.cloud.google.com     5
kredmo.com.au                       4
bankwest.com.au                     4
savingsmate.com.au                  3
commbank.com.au                     2
asic.gov.au                         2
bankmate.app                        2
Name: count, dtype: int64

In [24]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv()

# The schema you built — unchanged.
schema = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "description": "Every brand or organisation named in the answer.",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Name of the entity."},
                    "isCompetitor": {"type": "boolean", "description": "Whether the entity is a competitor to the brand being analysed."},
                    "sentiment": {
                        "type": "string",
                        "enum": ["positive", "neutral", "negative"],
                        "description": "Sentiment of the answer's coverage of this entity. Rate positive if brand is recommended highly or mentioned as the sole/most prominent example in a positive context. Rate neutral if they're listed/viable but not singled out in any way. Rate negative if they are criticised REGARDLESS of prominence."
                    }
                },
                "required": ["name", "isCompetitor", "sentiment"],
                "additionalProperties": False
            }
        }
    },
    "required": ["entities"],
    "additionalProperties": False
}


def analyse_answer(brand, answer):
    system_message = f"""You are analysing how an AI answer covers brands in a market.

The brand being analysed is: {brand}

Extract every brand or organisation named in the answer. For each one:
- name: the PARENT COMPANY or brand, normalised to its most common name. Do NOT return product or card names as separate entities — collapse them to the parent. For example: "Amex", "American Express Low Rate", and "Amex Explorer" all become "American Express". "ANZ Rewards Black" becomes "ANZ". "Bankwest Breeze Platinum" becomes "Bankwest". Use the shortest widely-recognised form of the brand.
- isCompetitor: true if it competes with {brand} in the same market, false otherwise (regulators, government bodies, and comparison/aggregator sites are not competitors)
- sentiment: how the answer treats that entity. positive = recommended, or named first and alone as the pick. neutral = listed as one of several viable options, not singled out. negative = criticised. If a brand is named first and alone as the recommendation, that is positive, not neutral. Only use neutral when a brand is one of several listed without clear preference.

If the same parent brand appears multiple times under different product names, return it ONCE with a single sentiment reflecting its overall treatment.

Only include entities actually named in the answer. Do not infer or add ones that aren't there."""

    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
        json={
            "model": "openai/gpt-5.6-luna",
            "temperature": 0,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": answer},
            ],
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "entity_analysis",
                    "strict": True,
                    "schema": schema,
                },
            },
        },
    )

    data = response.json()
    content = data["choices"][0]["message"]["content"]
    return json.loads(content)

In [25]:
row = results_df.loc[2]
result = analyse_answer("NAB", row["answer"])
result

{'entities': [{'name': 'NAB', 'isCompetitor': False, 'sentiment': 'neutral'},
  {'name': 'Reserve Bank of Australia',
   'isCompetitor': False,
   'sentiment': 'neutral'},
  {'name': 'ANZ', 'isCompetitor': True, 'sentiment': 'neutral'},
  {'name': 'Latitude', 'isCompetitor': True, 'sentiment': 'neutral'},
  {'name': 'Mastercard', 'isCompetitor': False, 'sentiment': 'neutral'},
  {'name': 'ING', 'isCompetitor': True, 'sentiment': 'neutral'},
  {'name': 'CommBank', 'isCompetitor': True, 'sentiment': 'neutral'},
  {'name': 'Westpac', 'isCompetitor': True, 'sentiment': 'neutral'},
  {'name': 'St.George', 'isCompetitor': True, 'sentiment': 'neutral'},
  {'name': 'Canstar', 'isCompetitor': False, 'sentiment': 'neutral'},
  {'name': 'Finder', 'isCompetitor': False, 'sentiment': 'neutral'},
  {'name': 'Mozo', 'isCompetitor': False, 'sentiment': 'neutral'},
  {'name': 'Money.com.au', 'isCompetitor': False, 'sentiment': 'neutral'}]}

In [26]:
import time

analysis_results = []   # collect in memory first

for idx, row in results_df.iterrows():
    print(f"Analysing row {idx}: {row['model_requested']}...")   # progress, so you see it working
    
    extracted = analyse_answer("NAB", row["answer"])   # the call you already have
    
    # attach provenance — which subject-model/question this came from
    analysis_results.append({
        "model_requested": row["model_requested"],
        "question": row["question"],
        "entities": extracted["entities"],
    })
    
    time.sleep(1)   # gentle pause so you don't hammer the API

print(f"\nDone — {len(analysis_results)} rows analysed")

Analysing row 0: openai/gpt-5.6-luna:online...
Analysing row 1: anthropic/claude-sonnet-5:online...
Analysing row 2: anthropic/claude-opus-5:online...
Analysing row 3: google/gemini-3.7-flash:online...
Analysing row 4: perplexity/sonar...
Analysing row 5: openai/gpt-5.6-luna:online...
Analysing row 6: anthropic/claude-sonnet-5:online...
Analysing row 7: anthropic/claude-opus-5:online...
Analysing row 8: google/gemini-3.7-flash:online...
Analysing row 9: perplexity/sonar...
Analysing row 10: openai/gpt-5.6-luna:online...
Analysing row 11: anthropic/claude-sonnet-5:online...
Analysing row 12: anthropic/claude-opus-5:online...
Analysing row 13: google/gemini-3.7-flash:online...
Analysing row 14: perplexity/sonar...
Analysing row 15: openai/gpt-5.6-luna:online...
Analysing row 16: anthropic/claude-sonnet-5:online...
Analysing row 17: anthropic/claude-opus-5:online...
Analysing row 18: google/gemini-3.7-flash:online...
Analysing row 19: perplexity/sonar...

Done — 20 rows analysed


In [27]:
import json

with open("nab_credit_cards_analysis.json", "w") as f:
    json.dump(analysis_results, f, indent=2)

print("Saved.")

Saved.


In [28]:
with open("nab_credit_cards_analysis.json") as f:
    check = json.load(f)
print(len(check), "rows loaded from disk")

20 rows loaded from disk


In [29]:
rows = []
for r in analysis_results:
    for e in r["entities"]:
        rows.append({
            "model_requested": r["model_requested"],
            "question": r["question"],
            "name": e["name"],
            "isCompetitor": e["isCompetitor"],
            "sentiment": e["sentiment"],
        })

entities_df = pd.DataFrame(rows)
print(entities_df.shape)
entities_df.head(15)

(132, 5)


,model_requested,question,name,isCompetitor,sentiment
0,openai/gpt-5.6-luna:online,I keep paying high interest on my credit card ...,Bankwest,True,neutral
1,openai/gpt-5.6-luna:online,I keep paying high interest on my credit card ...,Latitude,True,neutral
2,openai/gpt-5.6-luna:online,I keep paying high interest on my credit card ...,Canstar,False,neutral
3,openai/gpt-5.6-luna:online,I keep paying high interest on my credit card ...,Moneysmart,False,neutral
4,anthropic/claude-sonnet-5:online,I keep paying high interest on my credit card ...,Community First Bank,True,positive
5,anthropic/claude-sonnet-5:online,I keep paying high interest on my credit card ...,ING,True,positive
6,anthropic/claude-sonnet-5:online,I keep paying high interest on my credit card ...,Bankwest,True,positive
7,anthropic/claude-sonnet-5:online,I keep paying high interest on my credit card ...,ANZ,True,positive
8,anthropic/claude-sonnet-5:online,I keep paying high interest on my credit card ...,Money.com.au,False,neutral
9,anthropic/claude-sonnet-5:online,I keep paying high interest on my credit card ...,Mozo,False,neutral


In [30]:
competitors = entities_df[entities_df["isCompetitor"] == True]
competitors["name"].value_counts()

name
ANZ                          9
American Express             7
Westpac                      5
Bankwest                     4
ING                          4
Bank Australia               4
Community First Bank         3
Heritage Bank                3
Latitude                     2
CommBank                     2
Greater Bank                 1
Bank of us                   1
HSBC                         1
Qantas                       1
Virgin Australia             1
Health Professionals Bank    1
MyCard                       1
Coastline                    1
Qudos Bank                   1
BankVic                      1
Name: count, dtype: int64

In [31]:
competitors.groupby("name")["sentiment"].value_counts()

name                       sentiment
ANZ                        neutral      5
                           negative     3
                           positive     1
American Express           neutral      4
                           positive     2
                           negative     1
Bank Australia             neutral      4
Bank of us                 neutral      1
BankVic                    neutral      1
Bankwest                   neutral      3
                           positive     1
Coastline                  positive     1
CommBank                   neutral      2
Community First Bank       neutral      2
                           positive     1
Greater Bank               neutral      1
HSBC                       neutral      1
Health Professionals Bank  neutral      1
Heritage Bank              neutral      2
                           positive     1
ING                        neutral      3
                           positive     1
Latitude                   neutral     

In [32]:
entities_df[entities_df["name"] == "NAB"]["sentiment"].value_counts()

sentiment
neutral     4
negative    1
Name: count, dtype: int64